In [1]:
import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv("../data/raw/fraudTest.csv", index_col=0)
df = pd.DataFrame(data)
df

,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
,,,,,,,,,,,,,,,,,,,,,
0,2020-06-21 12:14:25,2291163933867244,fraud_Kirlin and Sons,personal_care,2.86,Jeff,Elliott,M,351 Darlene Green,Columbia,...,33.9659,-80.9355,333497,Mechanical engineer,1968-03-19,2da90c7d74bd46a0caf3777415b3ebd3,1371816865,33.986391,-81.200714,0
1,2020-06-21 12:14:33,3573030041201292,fraud_Sporer-Keebler,personal_care,29.84,Joanne,Williams,F,3638 Marsh Union,Altonah,...,40.3207,-110.4360,302,"Sales professional, IT",1990-01-17,324cc204407e99f51b0d6ca0055005e7,1371816873,39.450498,-109.960431,0
2,2020-06-21 12:14:53,3598215285024754,"fraud_Swaniawski, Nitzsche and Welch",health_fitness,41.28,Ashley,Lopez,F,9333 Valentine Point,Bellmore,...,40.6729,-73.5365,34496,"Librarian, public",1970-10-21,c81755dbbbea9d5c77f094348a7579be,1371816893,40.495810,-74.196111,0
3,2020-06-21 12:15:15,3591919803438423,fraud_Haley Group,misc_pos,60.05,Brian,Williams,M,32941 Krystal Mill Apt. 552,Titusville,...,28.5697,-80.8191,54767,Set designer,1987-07-25,2159175b9efe66dc301f149d3d5abf8c,1371816915,28.812398,-80.883061,0
4,2020-06-21 12:15:17,3526826139003047,fraud_Johnston-Casper,travel,3.19,Nathan,Massey,M,5783 Evan Roads Apt. 465,Falmouth,...,44.2529,-85.0170,1126,Furniture designer,1955-07-06,57ff021bd3f328f8738bb535c302a31b,1371816917,44.959148,-85.884734,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
555714,2020-12-31 23:59:07,30560609640617,fraud_Reilly and Sons,health_fitness,43.77,Michael,Olson,M,558 Michael Estates,Luray,...,40.4931,-91.8912,519,Town planner,1966-02-13,9b1f753c79894c9f4b71f04581835ada,1388534347,39.946837,-91.333331,0
555715,2020-12-31 23:59:09,3556613125071656,fraud_Hoppe-Parisian,kids_pets,111.84,Jose,Vasquez,M,572 Davis Mountains,Lake Jackson,...,29.0393,-95.4401,28739,Futures trader,1999-12-27,2090647dac2c89a1d86c514c427f5b91,1388534349,29.661049,-96.186633,0
555716,2020-12-31 23:59:15,6011724471098086,fraud_Rau-Robel,kids_pets,86.88,Ann,Lawson,F,144 Evans Islands Apt. 683,Burbank,...,46.1966,-118.9017,3684,Musician,1981-11-29,6c5b7c8add471975aa0fec023b2e8408,1388534355,46.658340,-119.715054,0


In [3]:
df_copy = df.copy()
df_copy['dob'] = pd.to_datetime(df_copy['dob'])
df_copy["trans_date_trans_time"] = pd.to_datetime(df_copy["trans_date_trans_time"])
df_copy["age"] = (
    df_copy["trans_date_trans_time"].dt.year - df_copy["dob"].dt.year
    - (
        (df_copy["trans_date_trans_time"].dt.month < df_copy["dob"].dt.month)
        | (
            (df_copy["trans_date_trans_time"].dt.month == df_copy["dob"].dt.month)
            & (df_copy["trans_date_trans_time"].dt.day < df_copy["dob"].dt.day)
        )
    )
).astype(int)

In [4]:
print("Minimum Age:", df_copy["age"].min())
print("Maximum Age:", df_copy["age"].max())

negative_ages = df_copy[df_copy["age"] < 0]
print("Number of negative ages:", len(negative_ages))

unrealistic_ages = df_copy[df_copy["age"] > 120]
print("Number of ages > 120:", len(unrealistic_ages))

df_copy["age"].describe()


Minimum Age: 15
Maximum Age: 96
Number of negative ages: 0
Number of ages > 120: 0


count    555719.000000
mean         46.390496
std          17.432211
min          15.000000
25%          33.000000
50%          44.000000
75%          58.000000
max          96.000000
Name: age, dtype: float64

In [5]:
df_copy['transaction_hours'] = df_copy['trans_date_trans_time'].dt.hour
df_copy['transaction_day'] = df_copy['trans_date_trans_time'].dt.day
df_copy['transaction_month'] = df_copy['trans_date_trans_time'].dt.month
df_copy['transaction_weekday'] = df_copy['trans_date_trans_time'].dt.dayofweek
df_copy["is_night_transaction"] = ((df_copy["trans_date_trans_time"].dt.hour < 6) | (df_copy["trans_date_trans_time"].dt.hour >= 23)).astype(int) 
df_copy['is_night_transaction']

 
0         0
1         0
2         0
3         0
4         0
         ..
555714    1
555715    1
555716    1
555717    1
555718    1
Name: is_night_transaction, Length: 555719, dtype: int64

In [6]:
merchant_fraud_rate = (
    df_copy.groupby("merchant")["is_fraud"]
    .mean()
)

df_copy["merchant_fraud_rate"] = df_copy["merchant"].map(merchant_fraud_rate)

top_10 = merchant_fraud_rate.sort_values(ascending=False).head(10)

print(top_10)

merchant
fraud_Romaguera, Cruickshank and Greenholt    0.021739
fraud_Lemke-Gutmann                           0.021505
fraud_Mosciski, Ziemann and Farrell           0.020690
fraud_Heathcote, Yost and Kertzmann           0.020482
fraud_Rodriguez, Yost and Jenkins             0.019960
fraud_Medhurst PLC                            0.019430
fraud_Bashirian Group                         0.018987
fraud_Kris-Weimann                            0.018939
fraud_Heathcote LLC                           0.018703
fraud_Bednar Group                            0.018519
Name: is_fraud, dtype: float64


In [7]:
R = 6371

lat1 = np.radians(df_copy["lat"])
lon1 = np.radians(df_copy["long"])
lat2 = np.radians(df_copy["merch_lat"])
lon2 = np.radians(df_copy["merch_long"])

d_lat = lat2 - lat1
d_lon = lon2 - lon1
        
# Haversine formula
a = (np.sin(d_lat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(d_lon / 2) ** 2)
c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

# Distance in kilometers
df_copy["merchant_distance"] = R * c

print(df_copy["merchant_distance"].head())
df_copy["merchant_distance"].describe()


 
0     24.561462
1    104.925092
2     59.080078
3     27.698567
4    104.335106
Name: merchant_distance, dtype: float64


count    555719.000000
mean         76.104902
std          29.117079
min           0.123883
25%          55.286255
50%          78.179517
75%          98.520760
max         150.922504
Name: merchant_distance, dtype: float64

In [ ]:
df_copy = df_copy.sort_values(["cc_num", "trans_date_trans_time"])
df_copy["previous_avg_amt"] = (df_copy.groupby("cc_num")["amt"].transform(lambda x: x.expanding().mean().shift(1)))
df_copy["amount_difference"] = (df_copy["amt"] - df_copy["previous_avg_amt"])
df_copy[["previous_avg_amt", "amount_difference"]] = (df_copy[["previous_avg_amt", "amount_difference"]].fillna(0))

In [9]:
df_copy["merchant_visit_frequency"] = (df_copy.groupby(["cc_num", "merchant"]).cumcount() + 1)
df_copy['merchant_visit_frequency']

 
157       1
741       1
3047      1
4351      1
7695      1
         ..
552584    3
552892    1
553559    2
553560    1
553883    1
Name: merchant_visit_frequency, Length: 555719, dtype: int64

In [10]:
col_drp = [
    'merchant_fraud_rate',
'previous_avg_amt',
'amount_difference',
'merchant_visit_frequency',
]
df_copy = df_copy.drop(columns=col_drp)

In [11]:

df_copy.to_csv("../data/processed/fraud_features.csv",index=False)
df_copy.columns

Index(['trans_date_trans_time', 'cc_num', 'merchant', 'category', 'amt',
       'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'lat',
       'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time', 'merch_lat',
       'merch_long', 'is_fraud', 'age', 'transaction_hours', 'transaction_day',
       'transaction_month', 'transaction_weekday', 'is_night_transaction',
       'merchant_distance'],
      dtype='str')